# Understanding Python

    Created: Zachary Lane, June 2024
    Last Edited: Zachary Lane, July 2024

This tutorial is aimed at introducing students to Python and some of the basic file and array manipulation techniques that are typically used in astronomy.

Work your way through the tutorial, **reading each code cell carefully before running it**. This is a live document, all the cells can easily be edited, and re-run any time you wish.

### The basic ways to interact with a Jupyter notebook:

- A "cell" is one of these little grey boxes that has "In [ ]:" at the left
- Double-clicking on a cell, or pressing Enter on a cell, will make it "active" so you can interact with it
- Shift + Enter executes the code that's in a cell 
- You can also run a cell by clicking 'Run'

#### Modules

It is typically good practice to import all of the modules you use at the start as in the following cell

In [ ]:
# This cell imports all of the modules we need. Some of these are not used here -- they are for future use.

import os # A module for communicating with the operating system e.g. commands and files
import glob # A module for searching data files
import matplotlib.pyplot as plt # A plotting library
import numpy as np # Numerical Python, great for vectorised equations
import scipy # Scientific Python, great for algorithms and optimisation
import pandas as pd # DataFrames for organising data/tables
from astropy.io import fits # Astronomy Python for opening ".fits" files
from photutils.background import Background2D, MedianBackground # Fitting background surfaces to astronomical images
from copy import deepcopy # Copies data in memory

%matplotlib inline


In [ ]:
# You do not need to fully understand this function, this cell is used to load a dataset for practice.

def create_random_data(size_cutout = 250, num_stars = 16, num_hot_pix = 6,
                       stddev = 0.9, noise_floor = 0.3, data_noise = 7, 
                       plot = True, column_noise = True,
                       sky_angle = 90, sky_units = 'deg', seeing = None, dark_noise = 900):
    '''
    This function creates a random data array with stars and hot pixels. The stars are modelled as Gaussian distributions
    
    Parameters
    ----------
    size_cutout : int, optional
        The size of the array. The default is 250.
    num_stars : int, optional
        The number of stars in the cutout. The default is 16.
    num_hot_pix : int, optional
        The number of hot pixels in the cutout. The default is 6.
    stddev : float, optional
        Point Spread Function (PSF) Size. The default is 0.9.
    noise_floor : float, optional
        Changes the Point Spread Function (PSF) size of each star. The default is 0.3.
    data_noise : float, optional
        Creates the random background noise with a varying background. The default is 7.
    plot : bool, optional
        Whether plots are attached. The default is True.
    column_noise : bool, optional
        Whether a bad column is added to the cutout. The default is True.
    sky_angle : float, optional
        The observing angle. The default is 90, e.g. zenith.
    sky_units : float, optional
        What units are used for the sky altitude to determine the "seeing". The default is 'deg'. Options are 'deg' or 'rad'.
    seeing : float, optional
        A seeing full-width half maximum value. The default is None.
    dark_noise : float, optional
        The background noise introduced by dark current. The default is 900.

    Returns
    -------
    data : array
        Simulated data array.

    '''
    
    data = np.random.randint(dark_noise - data_noise, dark_noise + data_noise) + data_noise * np.random.randn(size_cutout, size_cutout) 
    # Create the background array
    
    # Define parameters for Gaussian models of stars
    min_amplitude = 4000
    max_amplitude = 45000

    hotpix_amplitude = [55000, 65000]
    
    random_locations = np.random.randint(0, size_cutout, size=(num_stars, 2)) # Generate random locations for the stars

    hot_pix_random_locations = np.random.randint(0, size_cutout, size=(num_hot_pix, 2)) # Generate random locations for the stars
    
    amplitudes = np.random.randint(min_amplitude, max_amplitude, size=num_stars) # Generate random amplitudes for the models

    if seeing is None:
        if (sky_units.lower() == 'deg') & (sky_angle >= 15):
            seeing = 1 / np.cos(np.pi/2 - np.radians(sky_angle))
        elif (sky_units.lower() == 'rad') & (sky_angle >= np.pi/12):
            seeing = 1 / np.cos(np.pi/2 - sky_angle)
        else:
            seeing = 1
            print('Error: setting seeing to 1')
    
    for i in range(num_stars): # Generate Gaussian models and place them in the array
        sigma_x = seeing*abs(stddev + noise_floor * np.random.randn()) # Determine x std
        sigma_y = seeing*abs(stddev + noise_floor * np.random.randn()) # Determine y std
        x, y = random_locations[i] # Random locations of stars
        amplitude = amplitudes[i] # Getting the amplitude
        mean = [x, y] # Positions
        covariance = [[sigma_x, 0], [0, sigma_y]] # Covariance matrix for the fit
        gaussian_model = scipy.stats.multivariate_normal(mean=mean, cov=covariance) # Gaussian model
        x_indices = np.arange(0, size_cutout) # Placing x
        y_indices = np.arange(0, size_cutout) # Placing y
        xx, yy = np.meshgrid(x_indices, y_indices)
        xy = np.column_stack([xx.flat, yy.flat]) # Stacking x and y
        model_values = amplitude*gaussian_model.pdf(xy).reshape(xx.shape) # Gaussians probability distribution function
        
        noise = data_noise*np.random.randn(model_values.shape[0], model_values.shape[1]) # Add random noise to each pixel within the Gaussian model

        noisy_model_values = model_values + np.sqrt(model_values)*np.random.randn() # add noise and stars
        data += noisy_model_values # Create data with the stars
        
    if num_hot_pix: # Adding Hot pixels
        for j in range(num_hot_pix):
            data[hot_pix_random_locations[j][0], hot_pix_random_locations[j][1]] += np.random.randint(hotpix_amplitude[0], 
                                                                                                      hotpix_amplitude[1])
    if column_noise: # Adding a faulty column
        column_malfunction = data_noise * np.random.randn(data.shape[0]) + 15
        data[:, np.random.randint(0, data.shape[1])] += column_malfunction

    data = data.astype(int)

    if plot: # Plot the data
        plt.figure()
        plt.imshow(data, cmap = 'gray', vmin = np.nanpercentile(data, 2), vmax = np.nanpercentile(data, 98)) # 2d plotting
        plt.axis('off') # turning off axis labels
        plt.show()

    return data

def example_background(data):
    bkg_estimator = Background2D(data, (50, 50), filter_size=(5, 5), bkg_estimator=MedianBackground())
    median_background = bkg_estimator.background_median
    return median_background

# Practice datasets

We will use some practice datasets, each of these datasets are randomly generated so everyone should get different answers.

- Look at the two images that are output in the following cell, discuss the differences with the tutor before continuing.

In [ ]:
# This is the practice data that we will use for our analysis

data = create_random_data(num_stars = 31, sky_angle = 75)
data = create_random_data(num_stars = 31, sky_angle = 30)
data_example = deepcopy(create_random_data(plot = False))


# Using numpy and matplotlib

We are now going to practice array manipulation and plotting.

Choose a column of the "data_example" variable array at random and do the following...

- Plot the 1d slice.
- Plot the 1d slice as a histogram.
- Now plot the 2d image using "imshow" ... experiment with percentile and value vmin and vmax for displaying the images.  
- Plot a small subsection of the image (column 40 to 70 and row 120 to 150) with a different colourmap and different scaling.
- Plot a histogram of the 2d data subslice.

What is the difference between zooming in on our subsample and taking a slice?

Do you have axis labels?

Once you have all of those show them to the tutor to show understanding

# Statistics

From the "data_example" array variable find the following overall and per column. 

Plot the column value as a 1d plot.

- Minimum
- Maximum
- Median
- Mean
- Standard Deviation
- The 95th percentile

## Background

In astronomical detector technology, the "background" is typically referred to the overall optical "noise" of the system, that is, the incoming photons on the detector sensor in absence of photon sources.

Looking at the data and the histograms, what do you think the background value is? How did you come to this value? Subtract this value from "data_example" and plot the new value. Show this to a tutor an answer their questions about this.


## Comparison:

The following cell is a more robust estimate of the background of the data, how did your estimate compare?

This method uses kernel convolution and clipped statistics, if you want more information I would recommend discussing with a photometry expert


In [ ]:
bg = example_background(data_example)
print(bg)

# File Organising

Different operating systems run/find files in different ways. Both Linux and MacOS are built on Unix and therefore share similarities. Try avoid spaces and other specialty characters. 

- MacOS : /Users/zgl12/Research/ASTR211/
- Linux : /home/users/zgl12/Research/ASTR211/
- Windows : C:/zgl12/Research/ASTR211/ or C:\\\\zgl12\\\\Research\\\\ASTR211\\\\


Let us now see how we can find files and access them via python...

Change the folder paths to access the filesin the ASTR211/2024_Lab_Files/Phot_Skills_Lab/ from the MacDiarmid server (see General lab instructions).

In [ ]:
# Folders
folder = '/Users/zgl12/Research/ASTR211/Tut_Files/TYC/' # This is the direct folder with the files
folder_2 = '/Users/zgl12/Research/ASTR211/' # This is the master folder with the subfolders/files

# Goes through and finds each file in a designated folder

files = [folder + file for file in os.listdir(folder) if '.fit' in file]

files = [os.path.join(dirpath, filename) for dirpath, subdirs, filenames in os.walk(folder_2) for filename in filenames if ".fit" in filename]

print()
# Goes through and finds each file with a specific pattern in the folder, alternate method (puts in a list)
# This method is architecture dependent (e.g. the order for a MacOs M3 will be different to a Windows HP)
files = glob.glob(folder + '*.fit*')
print(files)